In [21]:
import requests
import pandas as pd
import time

SEARCH_QUERY = "lang:en -is:digital has:usd"
BASE_URL = "https://api.scryfall.com/cards/search"
DELAY = 0.1

def fetch_all_cards(query):
    all_cards = []
    url = BASE_URL
    params = {"q": query, "order": "released", "unique": "cards"}
    page = 1

    print("Starting data collection from Scryfall API...")

    while url:
        response = requests.get(url, params=params)
        params = None

        if response.status_code != 200:
            print(f"Error on page {page}: {response.status_code}")
            break

        data = response.json()
        cards = data.get("data", [])
        all_cards.extend(cards)

        url = data.get("next_page")
        page += 1
        time.sleep(DELAY)

    return all_cards

raw_cards = fetch_all_cards(SEARCH_QUERY)

Starting data collection from Scryfall API...


In [3]:
def parse_cards(cards):
    records = []

    for card in cards:
        prices = card.get("prices", {})

        record = {
            # Identifiers
            "name":             card.get("name"),
            "set_name":         card.get("set_name"),
            "released_at":      card.get("released_at"),

            # Card characteristics
            "cmc":              card.get("cmc"),
            "type_line":        card.get("type_line"),
            "rarity":           card.get("rarity"),
            "colors":           card.get("colors", []),
            "power":            card.get("power"),
            "toughness":        card.get("toughness"),
            "oracle_text":      card.get("oracle_text", ""),

            # Format legalities
            "legal_standard":   card.get("legalities", {}).get("standard"),
            "legal_modern":     card.get("legalities", {}).get("modern"),
            "legal_legacy":     card.get("legalities", {}).get("legacy"),
            "legal_commander":  card.get("legalities", {}).get("commander"),

            # Prices
            "price_usd":        prices.get("usd"),
            "price_usd_foil":   prices.get("usd_foil"),

            # Metadata
            "reprint":          card.get("reprint"),
            "promo":            card.get("promo"),
        }
        records.append(record)

    df = pd.DataFrame(records)

    # Type conversions
    df["price_usd"]      = pd.to_numeric(df["price_usd"],      errors="coerce")
    df["price_usd_foil"] = pd.to_numeric(df["price_usd_foil"], errors="coerce")
    df["cmc"]            = pd.to_numeric(df["cmc"],             errors="coerce")
    df["released_at"]    = pd.to_datetime(df["released_at"],    errors="coerce")
    df["release_year"]   = df["released_at"].dt.year

    # Engineered features
    df["power_numeric"]      = pd.to_numeric(df["power"],     errors="coerce")
    df["toughness_numeric"]  = pd.to_numeric(df["toughness"], errors="coerce")
    df["color_count"]        = df["colors"].apply(len)
    df["oracle_word_count"]  = df["oracle_text"].fillna("").apply(lambda x: len(x.split()))

    # Clean up
    df = df.dropna(subset=["price_usd"])
    df = df[~df["name"].isin(["Plains", "Island", "Swamp", "Mountain", "Forest"])]

    # Sort by price descending
    df = df.sort_values("price_usd", ascending=False).reset_index(drop=True)

    print(f"DataFrame ready: {df.shape[0]} rows x {df.shape[1]} columns")
    return df

df = parse_cards(raw_cards)

DataFrame ready: 31074 rows x 23 columns


In [ ]:
df_final = (
    df[["name", "price_usd", "price_usd_foil", "colors",
              "cmc", "legal_commander", "rarity", "type_line", "released_at"]]
    .sort_values("price_usd", ascending=False)
    .head(1000)
    .reset_index(drop=True)
    .rename(columns={
        "name":             "card_name",
        "price_usd":        "card_price",
        "price_usd_foil":   "foil_price",
        "colors":           "colors",
        "cmc":              "mana_cost",
        "legal_commander":  "commander_legal",
        "type_line":        "card_type",
        "released_at":      "release_year"
    })
)

# Then extract just the year
df_final["release_year"] = pd.to_datetime(df_final["release_year"]).dt.year

In [16]:
df_final.to_csv("mtg_top1000.csv", index=False)
print("Saved to mtg_top1000.csv")

Saved to mtg_top1000.csv


In [20]:
# -------------------------------------------------------
# Q1: Most and Least Expensive Card
# -------------------------------------------------------
 
print("\n--- Q1: Most & Least Expensive Card ---")
 
most_expensive = df_final.loc[df_final["card_price"].idxmax()]
least_expensive = df_final.loc[df_final["card_price"].idxmin()]
 
print(f"Most expensive:  {most_expensive['card_name']} — ${most_expensive['card_price']:.2f}")
print(f"Least expensive: {least_expensive['card_name']} — ${least_expensive['card_price']:.2f}")
 
 
# -------------------------------------------------------
# Q2: Average Price — Commander Legal vs Banned
# -------------------------------------------------------
 
print("\n--- Q2: Avg Price by Commander Legality ---")
 
commander_avg = (
    df_final.groupby("commander_legal")["card_price"]
    .agg(avg_price="mean", count="count")
    .round(2)
)
print(commander_avg)
 
 
# -------------------------------------------------------
# Q3: Most Expensive Color on Average
# -------------------------------------------------------
 
print("\n--- Q3: Average Price by Color ---")
 
# Colors is stored as a list (e.g. ['R', 'G']), so we need to explode it.
# First, evaluate the string representation of the list back to an actual list.
df_final["colors_list"] = df_final["colors"]
 
# Explode so each color gets its own row, then group
color_avg = (
    df_final.explode("colors_list")
    .groupby("colors_list")["card_price"]
    .agg(avg_price="mean", count="count")
    .sort_values("avg_price", ascending=False)
    .round(2)
)
 
# Map color codes to full names for readability
color_names = {"W": "White", "U": "Blue", "B": "Black", "R": "Red", "G": "Green"}
color_avg.index = color_avg.index.map(color_names)
 
print(color_avg)
 
# Also check colorless cards separately
colorless = df_final[df_final["colors_list"].apply(len) == 0]
print(f"\nColorless cards — avg price: ${colorless['card_price'].mean():.2f} (n={len(colorless)})")
 
 
# -------------------------------------------------------
# Q4: How Does CMC Affect Price?
# -------------------------------------------------------
 
print("\n--- Q4: Average Price by CMC ---")
 
cmc_avg = (
    df_final.groupby("mana_cost")["card_price"]
    .agg(avg_price="mean", count="count")
    .sort_values("mana_cost")
    .round(2)
)
print(cmc_avg)
 
# Correlation between CMC and price
correlation = df_final[["mana_cost", "card_price"]].corr().iloc[0, 1]
print(f"\nCorrelation between CMC and card price: {correlation:.3f}")


--- Q1: Most & Least Expensive Card ---
Most expensive:  Timetwister — $5142.02
Least expensive: Watery Grave — $11.86

--- Q2: Avg Price by Commander Legality ---
                 avg_price  count
commander_legal                  
banned             1046.29     19
legal                66.20    966
not_legal            48.40     15

--- Q3: Average Price by Color ---
             avg_price  count
colors_list                  
Blue             99.10    152
Black            66.69    193
White            48.22    148
Green            44.59    187
Red              42.12    149

Colorless cards — avg price: $131.35 (n=316)

--- Q4: Average Price by CMC ---
           avg_price  count
mana_cost                  
0.0           231.20    129
1.0            75.65     68
2.0            89.44    137
3.0            66.32    194
4.0            71.94    202
5.0            37.16    116
6.0            43.32     60
7.0            27.01     38
8.0            32.29     28
9.0            61.34     12
10.

In [19]:
# -------------------------------------------------------
# Q5: How Does Foiling Affect Price?
# -------------------------------------------------------
 
print("\n--- Q5: Foil vs Non-Foil Price ---")
 
# Only use rows where foil price is available
df_foil = df_final.dropna(subset=["foil_price"]).copy()
print(f"Cards with foil price available: {len(df_foil)}")
 
avg_non_foil = df_foil["card_price"].mean()
avg_foil     = df_foil["foil_price"].mean()
avg_premium  = (df_foil["foil_price"] - df_foil["card_price"]).mean()
avg_multiplier = (df_foil["foil_price"] / df_foil["card_price"]).mean()
 
print(f"Avg non-foil price:  ${avg_non_foil:.2f}")
print(f"Avg foil price:      ${avg_foil:.2f}")
print(f"Avg foil premium:    +${avg_premium:.2f}")
print(f"Avg foil multiplier: {avg_multiplier:.2f}x")


--- Q5: Foil vs Non-Foil Price ---
Cards with foil price available: 549
Avg non-foil price:  $27.68
Avg foil price:      $59.48
Avg foil premium:    +$31.80
Avg foil multiplier: 1.95x


In [18]:
print("\n--- Q6: Release Year Stats ---")

median_year = df_final["release_year"].median()
min_year    = df_final["release_year"].min()
max_year    = df_final["release_year"].max()

print(f"Median release year: {int(median_year)}")
print(f"Year range: {int(min_year)} — {int(max_year)}")

# Bonus: see how many cards come from each year
year_counts = (
    df_final.groupby("release_year")["card_price"]
    .agg(count="count", avg_price="mean")
    .sort_index()
    .round(2)
)
print("\nCards per year:")
print(year_counts)


--- Q6: Release Year Stats ---
Median release year: 2019
Year range: 1993 — 2026

Cards per year:
              count  avg_price
release_year                  
1993             43     820.01
1994             91     193.91
1995              8      16.44
1996             14      90.84
1997             33      46.20
1998             35     140.49
1999             90      72.31
2000              9      25.12
2001              7      17.69
2002             10      15.35
2003              7      24.39
2004             16      23.83
2005              9      18.27
2006              7      18.76
2007             12      19.21
2008             12      27.51
2009              7      15.21
2010              9      18.82
2011              7      20.78
2012              1      31.32
2013              7      21.40
2014              9      17.56
2015              4      30.75
2016             17      20.39
2017             14      21.41
2018             21      25.42
2019             15      21.47
20